# Dane zewnętrzne: stopy procentowe i wynagrodzenia

Dwa publiczne źródła uzupełniają ogłoszenia mieszkaniowe:

| Dane | Źródło | Do czego służą |
|---|---|---|
| Stopa referencyjna NBP | [Narodowy Bank Polski](https://nbp.pl/podstawowe-stopy-procentowe-archiwum/) – archiwum stóp procentowych w XML | miesięczna rata kredytu („kupić czy wynająć”) |
| Przeciętne miesięczne wynagrodzenie brutto w 15 miastach | [GUS – Bank Danych Lokalnych (BDL)](https://bdl.stat.gov.pl), publiczne API | dostępność mieszkań („na co mnie stać”) |

Oba źródła są pobierane automatycznie – bez ręcznie ściąganych plików. Surowe odpowiedzi trafiają do `data/raw/external/`, czyste tabele do `data/clean/`.

Kod jest w [`external.py`](external.py).

In [1]:
from pathlib import Path

import pandas as pd

import external as ext
from clean import CITIES

RAW_EXT_DIR = Path("../data/raw/external")
CLEAN_DIR = Path("../data/clean")
RAW_EXT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

## Stopa referencyjna NBP
Archiwum zawiera każdą zmianę stóp NBP. Zamieniamy je na **średnią miesięczną** (ważoną liczbą dni obowiązywania każdej stopy), żeby dało się ją połączyć z miesięcznymi zrzutami ogłoszeń.

In [2]:
xml_bytes = ext.fetch_nbp_rates_xml(save_to=RAW_EXT_DIR / "nbp_interest_rates.xml")
changes = ext.parse_nbp_rate_changes(xml_bytes)
changes.tail(8)

Downloaded 37,101 bytes from https://static.nbp.pl/dane/stopy/stopy_procentowe_archiwum.xml


,valid_from,rate_pct
88,2023-10-05,5.75
89,2025-05-08,5.25
90,2025-07-03,5.00
91,2025-09-04,4.75
92,2025-10-09,4.50
93,2025-11-06,4.25
94,2025-12-04,4.00
95,2026-03-05,3.75


In [3]:
rates = ext.monthly_average_rate(changes, start="2023-01-01", end=pd.Timestamp.today())
project_period = rates["month"].between("2023-08-01", "2024-06-01")
rates.loc[project_period]

,month,reference_rate_pct
7,2023-08-01,6.750
8,2023-09-01,6.150
9,2023-10-01,5.782
10,2023-11-01,5.750
11,2023-12-01,5.750
12,2024-01-01,5.750
13,2024-02-01,5.750
14,2024-03-01,5.750
15,2024-04-01,5.750
16,2024-05-01,5.750


**Jak jest używana:** dashboard szacuje oprocentowanie kredytu jako *stopa referencyjna NBP + marża banku*. Marża to parametr what-if w Power BI (domyślnie 2 pkt proc.), bo realne oferty zależą od banku, kredytobiorcy i wskaźnika WIBOR/WIRON – stopa referencyjna wyznacza tylko ogólny poziom.

## Przeciętne wynagrodzenie brutto w 15 miastach
Zmienna BDL `64428` – *przeciętne miesięczne wynagrodzenie brutto, ogółem* – na poziomie powiatów. Wszystkie 15 miast to miasta na prawach powiatu, więc każde ma własną wartość. Najpierw sprawdzenie, czy numer zmiennej wskazuje właściwy wskaźnik:

In [4]:
info = ext.fetch_bdl_variable_info()
{k: v for k, v in info.items() if k in ("id", "subjectId", "n1", "n2", "n3", "measureUnitName", "level")}

{'id': 64428,
 'subjectId': 'P2497',
 'n1': 'ogółem',
 'level': 5,
 'measureUnitName': 'zł'}

In [5]:
results = ext.fetch_bdl_by_variable(
    ext.BDL_WAGE_VARIABLE, years=[2022, 2023, 2024], unit_level=ext.BDL_POWIAT_LEVEL,
    save_to=RAW_EXT_DIR / "bdl_wages_powiats.json",
)
wages = ext.parse_bdl_wages(results)
wages.pivot(index="city_key", columns="year", values="avg_gross_wage_pln")

Downloaded 380 territorial units


year,2022,2023,2024
city_key,,,
bialystok,6183.98,7089.31,8281.89
bydgoszcz,6435.59,7478.19,8448.23
czestochowa,6018.76,6715.44,7830.22
gdansk,7876.68,8915.97,10107.32
gdynia,6985.73,7890.40,8999.18
katowice,8017.49,9105.52,10051.54
krakow,8157.43,9223.73,10455.44
lodz,6629.23,7548.10,8621.77
lublin,6402.56,7443.57,8507.48


Uwagi do tego wskaźnika:
- to wynagrodzenie **brutto**; dashboard przelicza dochód na netto prostym parametrem,
- na poziomie powiatów GUS podaje wynagrodzenia **bez podmiotów zatrudniających do 9 osób**, więc to punkt odniesienia dla typowej pensji na pełny etat, a nie dochód każdego gospodarstwa domowego,
- ogłoszenia obejmują okres sierpień 2023 – czerwiec 2024, więc każdy miesiąc łączymy z wynagrodzeniem z jego roku.

## Zapis

In [6]:
wages.insert(1, "city_name", wages["city_key"].map(lambda k: CITIES[k][0]))
rates.to_csv(CLEAN_DIR / "nbp_reference_rate.csv", index=False, date_format="%Y-%m-%d")
wages.to_csv(CLEAN_DIR / "gus_wages.csv", index=False)
pd.DataFrame({
    "file": ["nbp_reference_rate.csv", "gus_wages.csv"],
    "rows": [len(rates), len(wages)],
    "columns": [rates.shape[1], wages.shape[1]],
})

,file,rows,columns
0,nbp_reference_rate.csv,47,2
1,gus_wages.csv,45,6
